<a href="https://colab.research.google.com/github/dgaida/ba_hcai_th_public/blob/main/assets/exercises/04_bias_embeddings_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Übung: Bias in der Sprache – Was lernen Embedding-Modelle über unsere Welt?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<GITHUB_USER>/<REPO>/blob/main/<PFAD>/HCAI_03_Uebung_bias_embeddings_student.ipynb)

**Modul:** Mensch-zentrierte Künstliche Intelligenz · **Lernraum II:** Bias & Fairness

---

## Worum geht es?

*Embedding-Modelle* wandeln Text in Vektoren um (vgl. RAG-Vorlesung). Sie stecken in Suchmaschinen, Empfehlungssystemen, RAG-Chatbots und Bewerber-Matching. Trainiert werden sie auf riesigen Textsammlungen aus dem Internet. Dabei lernen sie nicht nur Sprache, sondern auch die **Stereotype, die in diesen Texten stecken**.

Wir messen das mit der **Cosine Similarity**. Die Idee stammt aus dem *WEAT-Test* (Caliskan et al., 2017): Ist ein Zielwort (z. B. „nurse") im Vektorraum näher an „woman" als an „man"?

**Leitfrage der Übung:** *Welche Vorurteile steckt ein Embedding-Modell, wie messen wir sie, und was bedeutet das für Anwendungen, die darauf aufbauen?*

## Lernziele

Sie können …
1. die Cosine Similarity berechnen und als Maß für semantische Nähe interpretieren,
2. mit einem einfachen Assoziationstest Bias in Embeddings messen (Geschlecht, Herkunft, Alter, …),
3. gefundene Verzerrungen einem **Bias-Typ** aus der Vorlesung zuordnen (Sampling, Label, Measurement, Historical, Algorithmic) und die Ursache in **Daten oder Modell** verorten,
4. die Auswirkung auf eine Anwendung (Ranking von Lebensläufen) abschätzen,
5. einen einfachen Debiasing-Ansatz anwenden und seine Grenzen kritisch bewerten.

## Ablauf

| Teil | Inhalt | Aufgaben |
|---|---|---|
| 0 | Setup: Modell von Hugging Face laden | – |
| 1 | Warm-up: Cosine Similarity und Wortanalogien | Aufgabe 1 |
| 2 | Bias bei Geschlecht und Berufen | Aufgabe 2 |
| 3 | Weitere Dimensionen: Herkunft, Alter, eigene Idee | Aufgabe 3 |
| 4 | Auswirkung: Bewerber-Ranking | Aufgabe 4 |
| 5 | Bias entfernen? Projektion der Geschlechtsrichtung | Aufgabe 5 |
| 6 | Zusatzaufgaben | optional |

> **Hinweise:** Die Code-Zellen sind lauffähig. Führen Sie sie der Reihe nach aus, lesen Sie die Ergebnisse und beantworten Sie die Fragen in den mit **✏️ Ihre Antwort** markierten Zellen (Doppelklick zum Bearbeiten).
> Die Ergebnisse hängen vom Modell ab. Auch „kein klarer Bias gefunden" ist ein Befund. Die Beispiele enthalten Stereotype. Wir untersuchen sie, um sie zu erkennen, nicht um sie zu bestätigen.

## Teil 0: Setup & Modell laden

Wir nutzen die Bibliothek `sentence-transformers`. Das Modell wird beim ersten Aufruf von Hugging Face heruntergeladen (einige Sekunden bis Minuten, je nach Modellgröße). Ein Account bzw. Token ist für öffentliche Modelle nicht nötig.

In [ ]:
!pip install -q -U sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

# Modellwahl: klein genug, um auf der Colab-CPU in Sekunden zu laufen.
MODELL_NAME = "BAAI/bge-small-en-v1.5"   # englisch, ca. 33 Mio. Parameter, ca. 130 MB
PRAEFIX = ""                             # manche Modelle erwarten einen Präfix (siehe unten)

# Alternativen (dann diese Zelle und alles danach neu ausführen):
#   "intfloat/multilingual-e5-small"   mehrsprachig (inkl. Deutsch), ca. 118 Mio. Parameter
#                                      -> PRAEFIX = "query: " setzen!
#   "sentence-transformers/all-MiniLM-L6-v2"   englisch, ca. 22 Mio. Parameter
#   "google/embeddinggemma-300m"       mehrsprachig, ca. 300 Mio. Parameter,
#                                      benötigt Hugging-Face-Token (Lizenz akzeptieren)

In [ ]:
model = SentenceTransformer(MODELL_NAME)
print(f"Modell: {MODELL_NAME}")
print(f"Dimension der Embedding-Vektoren: {model.get_sentence_embedding_dimension()}")

### Hilfsfunktionen (vorgegeben)

Lesen Sie den Code, Sie benötigen die Funktionen in allen folgenden Teilen. Der Kern der Bias-Messung ist `assoziation`: Für jedes Zielwort wird die mittlere Cosine Similarity zu den Wörtern der Attributgruppe A und der Attributgruppe B berechnet. Der **Bias-Wert** ist die Differenz `sim_B − sim_A`. Ein positiver Wert bedeutet: Das Zielwort liegt im Vektorraum näher an Gruppe B.

In [ ]:
def embed(texte, praefix=None):
    """Berechnet normalisierte Embedding-Vektoren.

    Args:
        texte (str | list[str]): Ein Text oder eine Liste von Texten.
        praefix (str | None): Präfix, der jedem Text vorangestellt wird.
            ``None`` bedeutet: der globale ``PRAEFIX`` aus der Konfiguration.

    Returns:
        np.ndarray: Matrix der Form (Anzahl Texte, Dimension). Jede Zeile hat
        die Länge 1, daher ist das Skalarprodukt zweier Zeilen ihre
        Cosine Similarity.
    """
    if isinstance(texte, str):
        texte = [texte]
    praefix = PRAEFIX if praefix is None else praefix
    return model.encode([praefix + t for t in texte],
                        normalize_embeddings=True, show_progress_bar=False)


def cosine(a, b):
    """Berechnet die Cosine Similarity cos(θ) = a·b / (|a|·|b|).

    Args:
        a (np.ndarray): Erster Vektor.
        b (np.ndarray): Zweiter Vektor.

    Returns:
        float: Wert zwischen -1 und 1 (in der Praxis meist zwischen 0 und 1).
    """
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def naechste_woerter(vektor, vokabular, top_n=5, ausschliessen=()):
    """Findet die Wörter eines Vokabulars, die einem Vektor am ähnlichsten sind.

    Args:
        vektor (np.ndarray): Suchvektor (z. B. das Ergebnis einer Vektorrechnung).
        vokabular (list[str]): Kandidatenwörter.
        top_n (int): Anzahl der zurückgegebenen Treffer.
        ausschliessen (tuple[str, ...]): Wörter, die nicht zurückgegeben werden
            (z. B. die Eingabewörter selbst).

    Returns:
        list[tuple[str, float]]: Paare (Wort, Cosine Similarity), absteigend sortiert.
    """
    sims = embed(vokabular) @ (vektor / np.linalg.norm(vektor))
    treffer = [(w, float(s)) for w, s in zip(vokabular, sims) if w not in ausschliessen]
    treffer.sort(key=lambda x: x[1], reverse=True)
    return treffer[:top_n]


def assoziation_aus_vektoren(ziele_vek, attr_a_vek, attr_b_vek, index, name_a="A", name_b="B"):
    """Misst, ob Zielvektoren näher an Attributgruppe A oder B liegen.

    Args:
        ziele_vek (np.ndarray): Normalisierte Vektoren der Zielwörter.
        attr_a_vek (np.ndarray): Normalisierte Vektoren der Attributgruppe A.
        attr_b_vek (np.ndarray): Normalisierte Vektoren der Attributgruppe B.
        index (list[str]): Namen der Zielwörter (Zeilenbeschriftung).
        name_a (str): Name der Gruppe A (Spaltenbeschriftung).
        name_b (str): Name der Gruppe B (Spaltenbeschriftung).

    Returns:
        pd.DataFrame: Spalten ``sim_<A>``, ``sim_<B>`` (mittlere Cosine Similarity)
        und ``bias`` = ``sim_<B>`` − ``sim_<A>``, aufsteigend nach ``bias`` sortiert.
    """
    sim_a = (ziele_vek @ attr_a_vek.T).mean(axis=1)
    sim_b = (ziele_vek @ attr_b_vek.T).mean(axis=1)
    df = pd.DataFrame({f"sim_{name_a}": sim_a, f"sim_{name_b}": sim_b,
                       "bias": sim_b - sim_a}, index=index)
    return df.sort_values("bias")


def assoziation(ziele, attribute_a, attribute_b, name_a="A", name_b="B"):
    """Assoziationstest (WEAT-Idee) für Texte statt Vektoren.

    Args:
        ziele (list[str]): Zielwörter bzw. -sätze, deren Bias gemessen wird.
        attribute_a (list[str]): Wörter bzw. Sätze der Attributgruppe A.
        attribute_b (list[str]): Wörter bzw. Sätze der Attributgruppe B.
        name_a (str): Name der Gruppe A.
        name_b (str): Name der Gruppe B.

    Returns:
        pd.DataFrame: Siehe ``assoziation_aus_vektoren``.
    """
    return assoziation_aus_vektoren(embed(ziele), embed(attribute_a), embed(attribute_b),
                                    ziele, name_a, name_b)


def plot_bias(df, titel, name_a, name_b):
    """Zeichnet die Bias-Werte als horizontales Balkendiagramm.

    Args:
        df (pd.DataFrame): Ergebnis von ``assoziation`` (Spalte ``bias``).
        titel (str): Titel des Plots.
        name_a (str): Name der Gruppe A (negative Seite, blau).
        name_b (str): Name der Gruppe B (positive Seite, orange).
    """
    farben = ["tab:blue" if w < 0 else "tab:orange" for w in df["bias"]]
    ax = df["bias"].plot.barh(color=farben, figsize=(7, 0.32 * len(df) + 1.2))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(titel)
    ax.set_xlabel(f"← näher an {name_a}  |  näher an {name_b} →")
    plt.tight_layout()
    plt.show()


def permutationstest(werte_a, werte_b, n_permutationen=10_000, seed=RANDOM_STATE):
    """Zweiseitiger Permutationstest auf Unterschied der Mittelwerte.

    Beantwortet die Frage: Könnte ein so großer Unterschied der Gruppenmittelwerte
    allein durch Zufall entstehen, wenn die Gruppenzugehörigkeit keine Rolle spielt?

    Args:
        werte_a (array-like): Bias-Werte der Gruppe A.
        werte_b (array-like): Bias-Werte der Gruppe B.
        n_permutationen (int): Anzahl zufälliger Umordnungen der Gruppenlabels.
        seed (int): Startwert des Zufallsgenerators.

    Returns:
        tuple[float, float]: Differenz der Mittelwerte (A − B) und p-Wert.
        Ein kleiner p-Wert (z. B. < 0,05) spricht gegen reinen Zufall.
    """
    rng = np.random.default_rng(seed)
    a, b = np.asarray(werte_a, dtype=float), np.asarray(werte_b, dtype=float)
    diff = a.mean() - b.mean()
    alle = np.concatenate([a, b])
    groesser = 0
    for _ in range(n_permutationen):
        rng.shuffle(alle)
        if abs(alle[:len(a)].mean() - alle[len(a):].mean()) >= abs(diff):
            groesser += 1
    return float(diff), groesser / n_permutationen

## Teil 1: Warm-up – Cosine Similarity und Wortanalogien

Zwei Texte mit ähnlicher Bedeutung haben Embedding-Vektoren mit ähnlicher Richtung, also eine Cosine Similarity nahe 1. Prüfen wir das an einigen Wortpaaren.

In [ ]:
PAARE = [
    ("cat", "kitten"), ("cat", "dog"), ("cat", "car"),
    ("doctor", "physician"), ("doctor", "nurse"), ("doctor", "banana"),
    ("king", "queen"), ("man", "woman"),
]
tabelle = pd.DataFrame(
    [(a, b, cosine(embed(a)[0], embed(b)[0])) for a, b in PAARE],
    columns=["Wort 1", "Wort 2", "Cosine Similarity"],
)
tabelle.sort_values("Cosine Similarity", ascending=False)

### Wortanalogien: „König − Mann + Frau = ?"

Klassische Embeddings (word2vec, 2013) wurden berühmt durch Vektorarithmetik: `king − man + woman ≈ queen`. Wir suchen das Wort aus einem kleinen Vokabular, das dem Ergebnisvektor am nächsten ist.

> Hinweis: Moderne Satz-Embedding-Modelle wurden nicht für Analogien trainiert. Die Ergebnisse sind daher weniger sauber als bei word2vec. Das Vokabular ist klein, es kann also nur aus diesen Wörtern wählen.

In [ ]:
VOKABULAR = [
    "king", "queen", "prince", "princess", "man", "woman", "boy", "girl",
    "father", "mother", "brother", "sister",
    "doctor", "nurse", "surgeon", "engineer", "programmer", "homemaker",
    "teacher", "secretary", "boss", "manager", "assistant", "pilot",
    "scientist", "librarian", "cook", "architect", "lawyer", "receptionist",
]


def analogie(a, b, c, top_n=3):
    """Löst die Analogie "a verhält sich zu b wie ? zu c" per Vektorarithmetik (a − b + c).

    Args:
        a (str): Ausgangswort.
        b (str): Wort, das "abgezogen" wird.
        c (str): Wort, das "addiert" wird.
        top_n (int): Anzahl der Treffer.

    Returns:
        list[tuple[str, float]]: Nächste Wörter des Vokabulars mit Cosine Similarity.
    """
    v = embed(a)[0] - embed(b)[0] + embed(c)[0]
    return naechste_woerter(v, VOKABULAR, top_n, ausschliessen=(a, b, c))


for a, b, c in [("king", "man", "woman"), ("doctor", "man", "woman"), ("programmer", "man", "woman")]:
    print(f"{a} − {b} + {c} ≈ {analogie(a, b, c)}")

### Aufgabe 1: Cosine Similarity und Analogien

**a)** Welche Wortpaare sind am ähnlichsten, welche am unähnlichsten? Ist `doctor`–`nurse` ähnlicher als `doctor`–`banana`? Warum?

**b)** Kann man aus dem Wert allein ablesen, ob zwei Wörter „ähnlich genug" sind (z. B. ab 0,5)? Begründen Sie.

**c)** Was liefert `king − man + woman`? Was liefern `doctor − man + woman` und `programmer − man + woman`? Welche **Annahme über die Welt** steckt in diesen Ergebnissen?

**d)** Warum ist das für eine Anwendung wie Bewerber-Suche oder RAG relevant?

**✏️ Ihre Antwort (Aufgabe 1):**

a) _..._
b) _..._
c) _..._
d) _..._

## Teil 2: Bias bei Geschlecht und Berufen

Wir testen, ob Berufsbezeichnungen im Vektorraum näher an „männlichen" oder „weiblichen" Wörtern liegen.

- **Attributgruppe A:** männliche Wörter, **Gruppe B:** weibliche Wörter
- **Ziele:** 12 Berufe mit (laut grober Einschätzung) hohem Männeranteil und 12 mit hohem Frauenanteil
- **Bias-Wert** = `sim_weiblich − sim_männlich`. Positiv bedeutet: näher an „weiblich".

In [ ]:
MAENNLICH = ["man", "male", "he", "him", "father", "brother", "boy"]
WEIBLICH = ["woman", "female", "she", "her", "mother", "sister", "girl"]

# Grobe Einschätzung nach typischem Geschlechteranteil. Prüfen Sie in Aufgabe 2a einzelne Berufe in der Statistik!
BERUFE_M = ["programmer", "mechanic", "engineer", "surgeon", "CEO", "pilot",
            "firefighter", "electrician", "carpenter", "architect", "lawyer", "police officer"]
BERUFE_W = ["nurse", "secretary", "receptionist", "kindergarten teacher", "hairdresser", "housekeeper",
            "cleaner", "librarian", "dental hygienist", "social worker", "nanny", "flight attendant"]
BERUFE = BERUFE_M + BERUFE_W

df_berufe = assoziation(BERUFE, MAENNLICH, WEIBLICH, "maennlich", "weiblich")
plot_bias(df_berufe, "Berufe: Nähe zu männlichen vs. weiblichen Wörtern", "männlich", "weiblich")
df_berufe

In [ ]:
bias_m = df_berufe.loc[BERUFE_M, "bias"]
bias_w = df_berufe.loc[BERUFE_W, "bias"]
diff, p = permutationstest(bias_m, bias_w)
print(f"Ø Bias 'männlich dominierte' Berufe: {bias_m.mean():+.3f}")
print(f"Ø Bias 'weiblich dominierte' Berufe: {bias_w.mean():+.3f}")
print(f"Differenz: {diff:+.3f}  (Permutationstest p = {p:.4f})")

### Dasselbe mit ganzen Sätzen

In Anwendungen stehen Berufe selten allein, sondern in Sätzen. Wir wiederholen den Test mit Satzvorlagen. Ist das Ergebnis stabil?

In [ ]:
VORLAGE_ZIEL = "This person's occupation is {}."
ATTR_M_SAETZE = ["This person is a man.", "He is a man.", "This is a male person."]
ATTR_W_SAETZE = ["This person is a woman.", "She is a woman.", "This is a female person."]

df_saetze = assoziation([VORLAGE_ZIEL.format(b) for b in BERUFE], ATTR_M_SAETZE, ATTR_W_SAETZE,
                        "maennlich", "weiblich")
df_saetze.index = [i.removeprefix("This person's occupation is ").removesuffix(".") for i in df_saetze.index]
plot_bias(df_saetze, "Berufe in Sätzen: Nähe zu männlichen vs. weiblichen Sätzen", "männlich", "weiblich")

rang_wort = df_berufe["bias"].rank()
rang_satz = df_saetze["bias"].reindex(df_berufe.index).rank()
print(f"Rangkorrelation Wort- vs. Satzversion (Spearman): {rang_wort.corr(rang_satz):.2f}")

### Aufgabe 2: Bias bei Geschlecht

**a)** Welche Berufe sind am stärksten „männlich", welche am stärksten „weiblich" assoziiert? Suchen Sie für **zwei bis drei Berufe** die tatsächlichen Geschlechteranteile in Deutschland (z. B. Bundesagentur für Arbeit, Statistisches Bundesamt). Bildet das Modell die Realität ab?

**b)** Wie stabil ist das Ergebnis zwischen Wort- und Satzversion (Rangkorrelation)? Was schließen Sie daraus für die Aussagekraft **eines einzelnen** Tests?

**c)** **Bias-Typ:** Welcher Typ aus der Vorlesung (Sampling, Label, Measurement, Historical, Algorithmic) beschreibt die Ursache am besten? Begründen Sie. Liegt die Ursache in den **Daten** oder im **Modell**? Wer hat hier „gelabelt"?

**d)** *Fair vs. korrekt:* Das Modell bildet den Sprachgebrauch im Internet zutreffend ab. Ist es deshalb „in Ordnung"? Für welche Anwendungen wäre diese Abbildung **erwünscht**, für welche **schädlich**?

**✏️ Ihre Antwort (Aufgabe 2):**

a) _..._
b) _..._
c) _..._
d) _..._

## Teil 3: Weitere Dimensionen von Bias in Sprache

Geschlecht ist nur eine Dimension. Dieselbe Methode funktioniert für jede Gruppe, für die man Wortlisten definieren kann.

### 3a) Herkunft: Vornamen

Angelehnt an Caliskan et al. (2017) und an Bewerbungsstudien (Bertrand & Mullainathan, 2004): Liegen Vornamen, die in den USA typischerweise weißen bzw. Schwarzen Personen zugeordnet werden, unterschiedlich nah an angenehmen bzw. unangenehmen Wörtern? Der Bias-Wert ist hier `sim_unangenehm − sim_angenehm`.

In [ ]:
NAMEN_GRUPPE_1 = ["Adam", "Harry", "Josh", "Roger", "Amanda", "Courtney", "Heather", "Katie"]
NAMEN_GRUPPE_2 = ["Jamel", "Jerome", "Darnell", "Malik", "Ebony", "Latisha", "Lakisha", "Yolanda"]
ANGENEHM = ["caress", "freedom", "health", "love", "peace", "cheer", "friend", "heaven"]
UNANGENEHM = ["agony", "terrible", "horrible", "nasty", "evil", "war", "awful", "failure"]

df_namen = assoziation(NAMEN_GRUPPE_1 + NAMEN_GRUPPE_2, ANGENEHM, UNANGENEHM, "angenehm", "unangenehm")
b1, b2 = df_namen.loc[NAMEN_GRUPPE_1, "bias"], df_namen.loc[NAMEN_GRUPPE_2, "bias"]
diff, p = permutationstest(b1, b2)
print(f"Ø Bias Gruppe 1: {b1.mean():+.3f} | Gruppe 2: {b2.mean():+.3f}")
print(f"Differenz (1 − 2): {diff:+.3f}  (Permutationstest p = {p:.4f})")
plot_bias(df_namen, "Vornamen: Nähe zu angenehmen vs. unangenehmen Wörtern", "angenehm", "unangenehm")

### 3b) Alter: Bewerber unterschiedlichen Alters

Wir beschreiben Bewerber:innen verschiedenen Alters und prüfen, ob sie näher an „modernen/dynamischen" oder „veralteten/starren" Eigenschaften liegen. Der Bias-Wert ist `sim_veraltet − sim_modern`.

In [ ]:
ALTER = [25, 35, 45, 55, 65]
ZIELE_ALTER = [f"a {a}-year-old applicant" for a in ALTER]
MODERN = ["modern", "innovative", "adaptable", "tech-savvy", "dynamic"]
VERALTET = ["outdated", "slow", "forgetful", "rigid", "old-fashioned"]

df_alter = assoziation(ZIELE_ALTER, MODERN, VERALTET, "modern", "veraltet")
df_alter = df_alter.reindex(ZIELE_ALTER)   # in Altersreihenfolge statt nach Bias sortiert
df_alter

### 3c) Ihre eigene Dimension

Definieren Sie eine **eigene** Fragestellung. Ideen: soziale Herkunft („Arbeiterkind" vs. „Akademikerkind"), Nationalität bzw. Länder, Religion, Behinderung, Dialekt, Studiengang („Informatik" vs. „Soziale Arbeit"), Intersektionalität („Frau und Schwarz" vs. „Frau"). Formulieren Sie **vorher** eine Hypothese. Das Beispiel unten dürfen Sie ersetzen.

> Gehen Sie verantwortungsvoll mit den Ergebnissen um: Ein Embedding-Test zeigt Muster in Texten, keine Eigenschaften von Menschen.

In [ ]:
# TODO: Ersetzen bzw. erweitern Sie die Listen (mindestens 5 Einträge je Liste).
ZIELE_EIGEN = ["a student from a working-class family", "a student from a wealthy family",
               "a first-generation student", "a student whose parents are professors"]
ATTRIBUTE_A = ["intelligent", "ambitious", "hardworking", "talented", "successful"]
ATTRIBUTE_B = ["lazy", "unmotivated", "incapable", "unreliable", "failing"]
NAME_A, NAME_B = "positiv", "negativ"

df_eigen = assoziation(ZIELE_EIGEN, ATTRIBUTE_A, ATTRIBUTE_B, NAME_A, NAME_B)
plot_bias(df_eigen, "Eigene Dimension", NAME_A, NAME_B)
df_eigen

### Aufgabe 3: Weitere Bias-Dimensionen

**a) Herkunft:** Wie unterscheiden sich die beiden Namensgruppen, und ist der Unterschied laut Permutationstest vom Zufall zu unterscheiden? Erweitern Sie die Namenslisten und wiederholen Sie den Test. Wie ändert sich der p-Wert? Warum ist die **Stichprobengröße** wichtig?

**b) Alter:** Wie verändern sich die Bias-Werte mit dem Alter? Gibt es einen monotonen Verlauf?

**c) Eigene Dimension:** Welche Hypothese hatten Sie, was kam heraus? Wurde Ihre Hypothese bestätigt?

**d) Bias-Typ:** Ordnen Sie jeden der drei Befunde (Herkunft, Alter, eigene Dimension) einem Bias-Typ aus der Vorlesung zu und begründen Sie kurz. Wo sehen Sie **Überschneidungen** mehrerer Typen? Bedenken Sie dabei, dass die Trainingstexte überwiegend aus bestimmten Sprachen, Ländern und Gruppen stammen.

**e) Inattentional Bias:** Warum fallen solche Muster Entwickler:innen oft nicht auf, die ein Embedding-Modell nur als Baustein nutzen (z. B. in einer RAG-Anwendung)?

**✏️ Ihre Antwort (Aufgabe 3):**

a) _..._
b) _..._
c) _..._
d) _..._
e) _..._

## Teil 4: Auswirkung auf eine Anwendung – Bewerber-Ranking

Embeddings werden in der Praxis genutzt, um Dokumente nach Ähnlichkeit zur Anfrage zu ranken (Suche, RAG, Matching). Wir simulieren das: Eine Stellenanfrage wird mit **identischen** Lebenslauf-Sätzen verglichen, die sich **nur im Vornamen** unterscheiden (Counterfactual-Test). Ein fairer Ranker dürfte hier keinen systematischen Unterschied zeigen.

In [ ]:
ANFRAGE = "We are looking for an experienced software engineer."
VORLAGEN = [
    "{} has 5 years of experience as a software engineer and a master's degree in computer science.",
    "{} is a mechanical engineer with 8 years of experience in automotive design.",
    "{} worked for 10 years as a data scientist and led a team of five.",
]
NAMEN_M = ["James", "Michael", "Daniel", "Robert"]
NAMEN_W = ["Mary", "Jennifer", "Sarah", "Emily"]

q = embed(ANFRAGE)[0]
zeilen = []
for i, vorlage in enumerate(VORLAGEN, start=1):
    s_m = embed([vorlage.format(n) for n in NAMEN_M]) @ q
    s_w = embed([vorlage.format(n) for n in NAMEN_W]) @ q
    zeilen.append({"Lebenslauf": i, "Ø Score männlich": s_m.mean(),
                   "Ø Score weiblich": s_w.mean(), "Differenz (w − m)": s_w.mean() - s_m.mean()})
pd.DataFrame(zeilen).set_index("Lebenslauf")

In [ ]:
# Ranking für Lebenslauf 1: alle Namen, identischer Text
kandidaten = [(n, "männlich") for n in NAMEN_M] + [(n, "weiblich") for n in NAMEN_W]
scores = embed([VORLAGEN[0].format(n) for n, _ in kandidaten]) @ q
ranking = pd.DataFrame({"Name": [n for n, _ in kandidaten],
                        "Geschlecht (Name)": [g for _, g in kandidaten],
                        "Score": scores}).sort_values("Score", ascending=False)
ranking["Rang"] = range(1, len(ranking) + 1)
ranking.set_index("Rang")

### Aufgabe 4: Bewerber-Ranking

**a)** Wie groß sind die Unterschiede, und in welche Richtung gehen sie? Ist die Richtung bei allen drei Lebensläufen gleich?

**b)** Die Unterschiede sind oft klein (z. B. in der zweiten Nachkommastelle). Ist das **vernachlässigbar**? Überlegen Sie: Ein System rankt 1000 Bewerbungen und zeigt nur die ersten 10. Was passiert, wenn ein Score-Unterschied bei der Platzierung zwischen zwei sonst gleichen Bewerbungen den Ausschlag gibt? Erweitern Sie den Test (mehr Namen, mehr Vorlagen) und prüfen Sie mit `permutationstest`, ob ein systematischer Effekt vorliegt. „Kein klarer Effekt" ist auch ein Ergebnis.

**c)** **Bias-Typ:** Der Bias entsteht im Embedding-Modell, wird aber zum Bias der *Anwendung*. Ordnen Sie ihn einem Typ zu. Wer ist verantwortlich: die Entwickler:innen des Modells oder die der Anwendung?

**d)** Welche **zwei Maßnahmen** würden Sie in einem solchen Matching-System ergreifen? Bedenken Sie: Reicht es, die Namen aus den Lebensläufen zu entfernen? (Stichwort Proxy-Features, vgl. Vorlesung.) Wie würden Sie den Erfolg Ihrer Maßnahme messen?

**✏️ Ihre Antwort (Aufgabe 4):**

a) _..._
b) _..._
c) _..._
d) _..._

## Teil 5: Bias entfernen? Die Geschlechtsrichtung herausrechnen

Bolukbasi et al. (2016) schlugen vor, im Vektorraum eine **Geschlechtsrichtung** zu bestimmen und aus allen *geschlechtsneutralen* Wörtern (z. B. Berufen) herauszurechnen (Projektion):

1. Richtung bestimmen: Mittelwert der Differenzvektoren definitorischer Paare (`woman − man`, `she − he`, …).
2. Für jedes neutrale Wort den Anteil entlang dieser Richtung abziehen und den Vektor neu normalisieren.

Wir prüfen mit **anderen** Wörtern (`PRUEF_*`) als bei der Richtungsbestimmung, ob der Bias verschwindet, denn sonst wäre der Test zirkulär.

In [ ]:
DEFINITIONSPAARE = [("woman", "man"), ("she", "he"), ("mother", "father"),
                    ("sister", "brother"), ("girl", "boy")]
PRUEF_M = ["gentleman", "husband", "son", "uncle"]
PRUEF_W = ["lady", "wife", "daughter", "aunt"]


def geschlechtsrichtung(paare):
    """Bestimmt die Geschlechtsrichtung als mittleren Differenzvektor definitorischer Paare.

    Args:
        paare (list[tuple[str, str]]): Paare (weiblich, männlich), z. B. ("woman", "man").

    Returns:
        np.ndarray: Einheitsvektor, der von "männlich" nach "weiblich" zeigt.
    """
    diffs = embed([w for w, _ in paare]) - embed([m for _, m in paare])
    richtung = diffs.mean(axis=0)
    return richtung / np.linalg.norm(richtung)


def entferne_richtung(vektoren, richtung):
    """Projiziert eine Richtung aus Vektoren heraus und normalisiert neu.

    Args:
        vektoren (np.ndarray): Matrix (Anzahl Wörter, Dimension).
        richtung (np.ndarray): Einheitsvektor der zu entfernenden Richtung.

    Returns:
        np.ndarray: Matrix gleicher Form, deren Zeilen keinen Anteil entlang
        ``richtung`` mehr besitzen und die Länge 1 haben.
    """
    bereinigt = vektoren - np.outer(vektoren @ richtung, richtung)
    return bereinigt / np.linalg.norm(bereinigt, axis=1, keepdims=True)


richtung = geschlechtsrichtung(DEFINITIONSPAARE)
berufe_vek = embed(BERUFE)
berufe_ent = entferne_richtung(berufe_vek, richtung)

pruef_m, pruef_w = embed(PRUEF_M), embed(PRUEF_W)
vorher = assoziation_aus_vektoren(berufe_vek, pruef_m, pruef_w, BERUFE, "maennlich", "weiblich")
nachher = assoziation_aus_vektoren(berufe_ent, pruef_m, pruef_w, BERUFE, "maennlich", "weiblich")

print(f"Ø |Bias| vorher:  {vorher['bias'].abs().mean():.3f}")
print(f"Ø |Bias| nachher: {nachher['bias'].abs().mean():.3f}")
plot_bias(nachher, "Nach dem Herausrechnen der Geschlechtsrichtung", "männlich", "weiblich")

### Zwei kritische Tests

**Test 1: Zielkonflikt.** Wenden wir die Projektion auf *alle* Wörter an, auch auf Wörter, die Geschlecht per Definition tragen (`mother`/`father`), verlieren wir erwünschte Information.

**Test 2: Steckt die Information noch drin?** Wir trainieren einen einfachen Klassifikator, der aus dem Vektor eines Berufs vorhersagen soll, ob er zur „männlichen" oder „weiblichen" Berufsgruppe gehört. Kann er das auch nach dem Debiasing noch (Gonen & Goldberg, 2019: *„Lipstick on a Pig"*)?

In [ ]:
# Test 1: Zielkonflikt
mf = embed(["mother", "father"])
mf_ent = entferne_richtung(mf, richtung)
print(f"Cosine(mother, father) original:       {cosine(mf[0], mf[1]):.3f}")
print(f"Cosine(mother, father) nach Projektion: {cosine(mf_ent[0], mf_ent[1]):.3f}")

# Test 2: Proxy-Information
y = np.array([0] * len(BERUFE_M) + [1] * len(BERUFE_W))
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
for name, X in [("Original", berufe_vek), ("Nach Projektion", berufe_ent)]:
    acc = cross_val_score(LogisticRegression(C=10, max_iter=1000), X, y, cv=cv).mean()
    print(f"Accuracy Berufsgruppe aus Vektor vorhersagen ({name}): {acc:.2f}")

### Aufgabe 5: Debiasing und seine Grenzen

**a)** Wie verändert sich der mittlere Bias (vorher/nachher)? Ist er „weg"?
**b)** Was zeigt Test 1? Warum darf man die Projektion nicht blind auf alle Wörter anwenden? Welcher Zielkonflikt aus der Vorlesung („Fairness vs. Korrektheit") taucht hier auf?
**c)** Was zeigt Test 2? Was bedeutet das für die Idee, einfach das sensible Merkmal zu entfernen (*Fairness through Unawareness*)? Welche Rolle spielen **Proxy-Features**?
**d)** Ordnen Sie die Maßnahme in die Pipeline ein: Ist das Debiasing eine Maßnahme **vor**, **während** oder **nach** dem Training? Welche Vor- und Nachteile hat sie gegenüber einer Änderung der Trainingsdaten?
**e)** Würden Sie dieses Debiasing in einem Bewerber-Matching-System einsetzen? Wie würden Sie vorher nachweisen, dass es hilft?

**✏️ Ihre Antwort (Aufgabe 5):**

a) _..._
b) _..._
c) _..._
d) _..._
e) _..._

## Teil 6: Zusatzaufgaben (optional)

### Zusatzaufgabe A: Deutsch und das generische Maskulinum
Wechseln Sie in Teil 0 zu `MODELL_NAME = "intfloat/multilingual-e5-small"` und `PRAEFIX = "query: "` und führen Sie alles ab dort neu aus. Untersuchen Sie dann, ob „Die Ärzte" (generisch gemeint) im Vektorraum näher an „Männern" liegt als „Die Ärztinnen und Ärzte".

In [ ]:
DE_SAETZE = ["Die Ärzte der Klinik", "Die Ärztinnen der Klinik", "Die Ärztinnen und Ärzte der Klinik",
             "Die Lehrer der Schule", "Die Lehrerinnen der Schule", "Die Lehrerinnen und Lehrer der Schule"]
df_de = assoziation(DE_SAETZE, ["Männer", "Herren", "er"], ["Frauen", "Damen", "sie"], "maennlich", "weiblich")
df_de

**✏️ Ihre Antwort (Zusatzaufgabe A):** Was zeigt das für das generische Maskulinum? Ist das ein Argument für oder gegen geschlechtergerechte Sprache in Trainingsdaten? Welcher Bias-Typ liegt vor?

_Ihre Antwort hier ..._

### Zusatzaufgabe B: Modellvergleich
Führen Sie Teil 2 mit zwei verschiedenen Modellen aus (z. B. `bge-small-en-v1.5` und `all-MiniLM-L6-v2`). Sind die Ergebnisse gleich? Ist ein größeres Modell weniger verzerrt? Berechnen Sie die Rangkorrelation der Bias-Werte beider Modelle.

**✏️ Ihre Antwort (Zusatzaufgabe B):**

_Ihre Antwort hier ..._

### Zusatzaufgabe C: Gesamtbild
Verfassen Sie ein „Bias-Audit-Fazit" für Ihr Modell in fünf Sätzen, wie Sie es einem Produktmanager vorlegen würden: Was haben Sie gemessen, was gefunden, wie sicher sind Sie, welche Risiken ergeben sich für welche Anwendung, welche Maßnahme empfehlen Sie?

**✏️ Ihre Antwort (Zusatzaufgabe C):**

_Ihre Antwort hier ..._

---
*Literatur: Bolukbasi et al. (2016), Man is to Computer Programmer as Woman is to Homemaker? · Caliskan, Bryson & Narayanan (2017), Semantics derived automatically from language corpora contain human-like biases, Science · Gonen & Goldberg (2019), Lipstick on a Pig · Bertrand & Mullainathan (2004), Are Emily and Greg More Employable than Lakisha and Jamal?*